<a href="https://colab.research.google.com/github/gabrieleduardo/public-package-manager-complex-network/blob/main/Mestrado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise das redes de package managers como redes complexas

## Importações do ambiente

In [1]:
!pip install python-igraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 42.8 MB/s eta 0:00:00


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import igraph as ig

import math
from typing import Any

import igraph as ig

In [2]:
# Faz a leitura das vértices
print(f"Iniciando leitura das vértices")
df_vertices = pd.read_csv('/content/data/vertices_nuget.csv', encoding='latin1')
print("Imprimindo head das vértices")
display(df_vertices.head())

# Faz a leitura das arestas
print(f"Iniciando leitura das arestas")
df_arestas = pd.read_csv('/content/data/arestas_nuget.csv', encoding='latin1')
print("Imprimindo head das arestas")
display(df_arestas.head())



Pipeline Configurado para o Ecossistema: 'nuget'
Reprocessamento Forçado: False
Iniciando leitura das vértices


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/vertices_nuget.csv'

In [4]:
import igraph as ig

# Converter IDs para string para garantir que sejam tratados como nomes simbólicos
df_arestas_str = df_arestas.astype(str)

# Criar o grafo. use_vids=False indica que as colunas contêm nomes (ID original), não índices internos
graph = ig.Graph.DataFrame(df_arestas_str, directed=True, use_vids=False)

# Criar mapeamento de ID (como string) para vertice_string
vertex_attrs = df_vertices.set_index(df_vertices['id_numerico'].astype(str))['vertice_string'].to_dict()

# Atribuir o atributo 'vertice_string' de forma otimizada
graph.vs['vertice_string'] = [vertex_attrs.get(name, "Unknown") for name in graph.vs['name']]

print(f"Grafo criado com {graph.vcount()} vértices e {graph.ecount()} arestas.")
print(graph.summary())

Grafo criado com 7897000 vértices e 31863216 arestas.
IGRAPH DN-- 7897000 31863216 -- 
+ attr: name (v), vertice_string (v)


In [11]:

def _safe_float(value: Any) -> float:
    if value is None:
        return float("nan")
    try:
        val = float(value)
        if math.isnan(val):
            return 0.0
        return val
    except (TypeError, ValueError):
        return float("nan")

def _degree_assortativity(graph: ig.Graph) -> float:
    if graph.vcount() < 2 or graph.ecount() == 0:
        return float("nan")
    return graph.assortativity_degree(directed=False)

In [10]:
from datetime import datetime
print(f"=== Iniciando compute_summary_metrics === [{datetime.now().strftime('%H:%M:%S')}]")
node_count = graph.vcount()
print(f"=== Número de nós: {node_count} === [{datetime.now().strftime('%H:%M:%S')}]")
edge_count = graph.ecount()
print(f"=== Número de arestas: {edge_count} === [{datetime.now().strftime('%H:%M:%S')}]")
density = graph.density() if node_count > 1 else 0.0
print(f"=== Densidade: {density} === [{datetime.now().strftime('%H:%M:%S')}]")
wcc_count = len(graph.connected_components(mode="weak")) if node_count > 0 else 0
print(f"=== Número de componentes fracamente conexas: {wcc_count} === [{datetime.now().strftime('%H:%M:%S')}]")
scc_count = len(graph.connected_components(mode="strong")) if node_count > 0 else 0
print(f"=== Número de componentes fortemente conexas: {scc_count} === [{datetime.now().strftime('%H:%M:%S')}]")
if node_count > 0:
    undirected = graph.as_undirected(mode="collapse")
    avg_clustering = _safe_float(undirected.transitivity_undirected())
else:
    avg_clustering = 0.0
print(f"=== Coeficiente de agrupamento médio: {avg_clustering} === [{datetime.now().strftime('%H:%M:%S')}]")
assortativity = _safe_float(_degree_assortativity(graph))
print(f"=== Assortatividade: {assortativity} === [{datetime.now().strftime('%H:%M:%S')}]")

=== Iniciando compute_summary_metrics === [16:26:49]
=== Número de nós: 7897000 === [16:26:49]
=== Número de arestas: 31863216 === [16:26:49]
=== Densidade: 5.109346857962791e-07 === [16:26:49]
=== Número de componentes fracamente conexas: 41495 === [16:26:53]
=== Número de componentes fortemente conexas: 7896999 === [16:27:00]
=== Coeficiente de agrupamento médio: 0.000616555822404085 === [16:27:29]
=== Iniciando degree_assortativity === [16:27:29]
=== Retornando degree_assortativity === [16:27:29]
=== Assortatividade: -0.04940501655263074 === [16:27:29]
=== Retornando compute_summary_metrics === [16:27:29]
